# Gaussian in the Bay

In this case, we solve for a Gaussian bump advecting with the currents.

## Formal problem formulation

We want to find depth-averaged concentration $c(x, t)$ as the solution to the depth-averaged advection equation.

$$
\begin{cases}
    \frac{\partial \iota}{\partial t} + \nabla \cdot (\iota \mathbf{v}) = 0 &x \in \Omega \\
    \iota = 0 &x \in \Gamma_{\mathrm{in}}
\end{cases}
$$

- The conserved quantity $\iota$ is concentration scaled by height, $\iota = c h$.
- The domain $\Omega$ is taken to be the whole mesh from example 2, including dry land. This naturally handles wetting/drying, because inactivated cells have no momentum, and the conserved quantity $\iota$, which can be interpreted as salt mass per infinitesimal column, remains fixed as a "salt flat." Concentration $c$, however, is not well-defined and blows up.
- The time range is taken to be 9 days, $(0, T) = (0, 777600)$.
- Inflow boundaries $\Gamma_{\mathrm{in}}$ are prescribed no concentration, $\iota_{\mathrm{in}} = 0$.
- The height $h = \eta + b$ and velocity $v$ fields are derived from fort.63 and fort.64 nodal data files. These values are interpolated both in space (using a Matplotlib triangulation) and time (using a linear interpolation).

$$
\begin{align}
\mathbf{v}(x, t) &= (\mathbf{v}_{b}(x) - \mathbf{v}_{a}(x)) \frac{t - t_{a}}{t_{b} - t_{a}} + \mathbf{v}_{a} \\
\\ &= \frac{t - t_{a}}{t_{b} - t_{a}} \mathbf{v}_{b} + \frac{t_{b} - t}{t_{b} - t_{a}} \mathbf{v}_{a} \\
&t \in [t_{a}, t_{b}]
\end{align}
$$

- Crucually, the metric tensor is not yet implemented. For now, longitude and latitude are treated as Cartesian coordinates.

## Imports and setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import adcircxtools as at
import adios4dolfinx as adx
import basix.ufl
import dolfinx as dx
import dolfinx.fem.petsc
import dolfinx.plot as dxp
import fenicsxtools as ft
import matplotlib.pyplot as plt
import matplotlib.tri
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import scipy.interpolate
import scipy.spatial
import ufl

## Problem Parameters

The problem is initialized with a Gaussian bump at the center. Note that this includes initialization on dry land.

In [3]:
# For initial condition
radius_earth = 6371.0 # km
pa_lon = -97.0611 # deg
pa_lat = 27.8339 # deg
x0 = pa_lon + 0.0 # Offset
y0 = pa_lat + 0.0 # Offset
r0 = np.rad2deg(1.0 / radius_earth) # deg
def iota0(x):
    return np.exp(-((x[0] - x0) ** 2 + (x[1] - y0) ** 2)/ (2 * r0 ** 2))

In [4]:
# Initial velocity
# Field at time 0 is (0, 0)
def v0(x):
    return np.zeros((2, x.shape[1]))

In [5]:
# Time stepping parameters
t_final = 6 * 3000 # seconds
dt = 1.0 # seconds
write_every = 600 # Snapshot every 10 minutes
fps = 6 # For plot gif

## Read in mesh and set up data buffers

In [6]:
# Get main filtered mesh
domain = adx.read_mesh('port_aransas.bp', MPI.COMM_WORLD)

In [7]:
# Get structures for slightly larger mesh
# Used for triangular interpolation
interp_coordinates = np.load('interp_coordinates.npy')
interp_elements = np.load('interp_elements.npy')
interp_node_map = np.load('interp_node_map.npy')
interp_element_map = np.load('interp_element_map.npy')

In [8]:
# Initialize triangulation for function interpolation
triangulation_interp = matplotlib.tri.Triangulation(
    interp_coordinates[:, 0],
    interp_coordinates[:, 1],
    interp_elements
)

In [9]:
# Boolean masks for filtering out data points
interp_node_mask = interp_node_map != -1
interp_element_mask = interp_element_map != -1

In [10]:
# Set up buffers
elevation_buffer = at.io.TimeSeriesBuffer('6hr.63')
elevation_buffer.open()
velocity_buffer = at.io.TimeSeriesBuffer('6hr.64')
velocity_buffer.open()

In [11]:
# Read in first record for initialization
(elevation_buffer_time,
 elevation_buffer_it,
 elevation_buffer_dat) = elevation_buffer.read_step(interp_node_mask)
(velocity_buffer_time,
 velocity_buffer_it,
 velocity_buffer_dat) = velocity_buffer.read_step(interp_node_mask)

In [12]:
# Make units consistent
# For now, convert m/s to deg/s
# 1 degree of horizontal (longitude) depends on the latitude
# 1 degree of vertical (latitude) is same distance
velocity_buffer_dat[:, 0] = np.rad2deg(velocity_buffer_dat[:, 0] / (radius_earth * np.cos(np.deg2rad(interp_coordinates[:, 1]))))
velocity_buffer_dat[:, 1] = np.rad2deg(velocity_buffer_dat[:, 1] / radius_earth)
velocity_buffer_dat

array([[ 0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00],
       ...,
       [-1.56516564e-07,  1.34789198e-06],
       [-2.08178533e-07,  1.31175787e-06],
       [-2.48809286e-07,  1.33258373e-06]], shape=(6687, 2))

In [13]:
# Set up interpolation function for initialization
# This has to be re-set every time velocity_buffer_dat is updated
u_velocity_interpolator = matplotlib.tri.LinearTriInterpolator(
    triangulation_interp,
    velocity_buffer_dat[:, 1]
)
v_velocity_interpolator = matplotlib.tri.LinearTriInterpolator(
    triangulation_interp,
    velocity_buffer_dat[:, 1]
)
def vel(x):
    return np.vstack((u_velocity_interpolator(x[0], x[1]), v_velocity_interpolator(x[0], x[1])))

## FEM formulation

In [14]:
# Use piecewise linear elements
V = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1))
W = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1, (domain.geometry.dim,)))

In [15]:
# Initialize scaled concentration iota
iota = dx.fem.Function(V, name='iota')
iota.interpolate(iota0)

In [16]:
# Initialize velocity field function
# Defined as interpolation between two field snapshots
va = dx.fem.Function(W)
vb = dx.fem.Function(W)
va.interpolate(v0)
vb.interpolate(vel)
t = dx.fem.Constant(domain, 0.0) # Current simulation time
ta = dx.fem.Constant(domain, 0.0) # First time in interpolation interval
tb = dx.fem.Constant(domain, velocity_buffer_time) # Second time in interpolation interval
v = (t - ta) / (tb - ta) * vb + (tb - t) / (tb - ta) * va

In [17]:
# Generate conservation law
equation = ft.equations.get_advection(
    domain=domain,
    U=iota,
    v=0.0001 * v
)

In [18]:
# Choose standard DG weak formulation
# Use LLF flux
formulation = ft.formulations.DGFormulation(
    equation=equation,
    trace_function=ft.fluxes.fluxn_upwind_scalar
)

## Solver formulation

In [19]:
# Build TS solver
# Plotter
# Function updater

In [20]:
# Mass form
M_form = dx.fem.form(formulation.mass_form)
M = dx.fem.petsc.assemble_matrix(M_form)
M.assemble()
M_solver = PETSc.KSP().create(domain.comm)
M_solver.setOperators(M)
M_solver.setType(PETSc.KSP.Type.PREONLY)
M_solver.getPC().setType(PETSc.PC.Type.LU)
M_solver.setFromOptions() # Allows command-line PETSc options to override the above
M_solver.setUp()

In [21]:
# Residual form
g_form = dx.fem.form(formulation.residual_form)
G_vec = dx.fem.petsc.create_vector(V)
b = dx.fem.petsc.create_vector(V)

In [22]:
# Set up right hand side
# Uses external variables: iota, b, M_solver
def rhs(ts, tt, uu, GG):
    """Right-hand side G of the general TS ODE.

    F(t, u, du/dt) = G(t, u)

    Arguments:
        ts: A PETSc time stepper object.
        tt: The current time.
        uu: The PETSc state vector.
        GG: The PETSc residual value.

    Returns:
        None: The function sets a new value for GG.
    """
    # In this case, G = M^-1 R(u)
    # So there is both an update of the residual and a matrix solve

    # Update my function iota from TS function x
    dx.fem.petsc.assign(uu, iota)
    iota.x.scatter_forward()
    
    # Residual update
    with b.localForm() as b_local:
        b_local.set(0.0) # Flush everything to 0
    dx.fem.petsc.assemble_vector(b, g_form) # Update using weak form
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    
    # Matrix solve
    # No need to update M from weak form
    # Result goes to g, the "return value"
    M_solver.solve(b, GG)
    GG.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)

In [23]:
# Create PETSc TS solver
ts = PETSc.TS().create(domain.comm)
ts.setType(PETSc.TS.Type.EULER)
ts.setTime(0.0)
ts.setTimeStep(dt)
ts.setMaxTime(t_final)
ts.setExactFinalTime(PETSc.TS.ExactFinalTime.MATCHSTEP)
ts.setRHSFunction(rhs, G_vec)
ts.setSolution(iota.x.petsc_vec)

In [24]:
# Set up plotting preliminaries
topology, cell_types, geometry = dxp.vtk_mesh(V)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)
# Numerical plotter
grid.point_data['iota'] = np.asarray(iota.x.array.real).copy()
plotter = pv.Plotter()
plotter.add_mesh(
    grid,
    scalars='iota',
    cmap='jet',
    clim=[0.0, 1.0],
    show_edges=False
)
plotter.view_xy()
plotter.open_gif('advection.gif', fps=fps)
def monitor(ts, step, tt, uu):
    if step % write_every != 0:
        return
    print(f'writing step {step} at time {tt}')
    # Make sure solution is updated
    dx.fem.petsc.assign(uu, iota)
    iota.x.scatter_forward()
    # Plotting from rank 0
    if domain.comm.rank == 0:
        grid.point_data['iota'] = np.asarray(iota.x.array.real).copy()
        plotter.write_frame()
ts.setMonitor(monitor)

In [25]:
# Set up update function for velocity
def pre_step(ts):
    time = ts.getTime() # Start time of the upcoming step
    # Update t for velocity interpolation
    t.value = time
    # If out of range, reset va, vb, ta, tb
    if time > tb.value:
        # Push b values to a
        va.x.array[:] = vb.x.array
        ta.value = tb.value
        # Read in new b values
        (velocity_buffer_time,
         velocity_buffer_it,
         velocity_buffer_dat) = velocity_buffer.read_step(interp_node_mask)
        # Make units consistent
        # For now, convert m/s to deg/s
        # 1 degree of horizontal (longitude) depends on the latitude
        # 1 degree of vertical (latitude) is same distance
        velocity_buffer_dat[:, 0] = np.rad2deg(velocity_buffer_dat[:, 0] / (radius_earth * np.cos(np.deg2rad(interp_coordinates[:, 1]))))
        velocity_buffer_dat[:, 1] = np.rad2deg(velocity_buffer_dat[:, 1] / radius_earth)
        u_velocity_interpolator = matplotlib.tri.LinearTriInterpolator(
            triangulation_interp,
            velocity_buffer_dat[:, 1]
        )
        v_velocity_interpolator = matplotlib.tri.LinearTriInterpolator(
            triangulation_interp,
            velocity_buffer_dat[:, 1]
        )
        def vel(x):
            return np.vstack((u_velocity_interpolator(x[0], x[1]), v_velocity_interpolator(x[0], x[1])))
        vb.interpolate(vel)
        tb.value = velocity_buffer_time
ts.setPreStep(pre_step)

In [26]:
# Finalize setup
ts.setFromOptions()

## Solve

In [27]:
# Solution loop
ts.solve(iota.x.petsc_vec)

writing step 0 at time 0.0
writing step 600 at time 600.0
writing step 1200 at time 1200.0
writing step 1800 at time 1800.0
writing step 2400 at time 2400.0
writing step 3000 at time 3000.0
writing step 3600 at time 3600.0
writing step 4200 at time 4200.0
writing step 4800 at time 4800.0
writing step 5400 at time 5400.0
writing step 6000 at time 6000.0
writing step 6600 at time 6600.0
writing step 7200 at time 7200.0
writing step 7800 at time 7800.0
writing step 8400 at time 8400.0
writing step 9000 at time 9000.0
writing step 9600 at time 9600.0
writing step 10200 at time 10200.0
writing step 10800 at time 10800.0
writing step 11400 at time 11400.0
writing step 12000 at time 12000.0
writing step 12600 at time 12600.0
writing step 13200 at time 13200.0
writing step 13800 at time 13800.0
writing step 14400 at time 14400.0
writing step 15000 at time 15000.0
writing step 15600 at time 15600.0
writing step 16200 at time 16200.0
writing step 16800 at time 16800.0
writing step 17400 at time 

## Visualization

In [28]:
# Close and save file
plotter.close()

## Cleanup

In [29]:
# Close buffers
elevation_buffer.close()
velocity_buffer.close()